# Step 3 (Colab) — Train Diffusion Policy on Lebai LM3 demos

Reads the LeRobot dataset from a tarball on Drive (you build the tarball locally and upload it once), extracts to Colab's local SSD for fast access, and writes checkpoints back to Drive.

**Before running**

1. Locally, run `python convert_local.py` and `python verify_local.py` to produce `result/local/lebai_duck_pick/` (~3.5 GB).
2. Pack it: `cd result && tar -czf lebai_duck_pick.tar.gz local/lebai_duck_pick`.
3. Upload `lebai_duck_pick.tar.gz` to `MyDrive/Lebai_train_DiffusionPolicy/` (drag-drop in Drive UI is fine).
4. In Colab: Runtime → Change runtime type → **GPU** (T4 minimum, A100 recommended).

**Smoke test first.** DP needs more steps than ACT to converge — a full 100k-step run takes hours on T4. Before committing, run with `NUM_STEPS=200` and confirm `loss` decreases. The §13 cell below has a `SMOKE_TEST` flag for this.

## 1. Install dependencies

Pinning `lerobot==0.5.1` — newer versions move APIs (PRD §7).

In [ ]:
!pip install -q 'lerobot==0.5.1' matplotlib

## 2. Mount Drive and configure paths

Set `HF_LEROBOT_HOME` to a local SSD path **before** importing `lerobot`. Do NOT point this at Drive — Drive's FUSE layer corrupts thousands of small parquet shard reads (PRD pitfall #1).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
import subprocess
from pathlib import Path

DRIVE_ROOT     = Path('/content/drive/MyDrive/Lebai_train_DiffusionPolicy')
LEROBOT_CACHE  = Path('/content/lerobot_cache')              # Colab SSD — fast, ephemeral
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints/dp_run01'         # persisted on Drive
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_LEROBOT_HOME'] = str(LEROBOT_CACHE)

# Extract the dataset tarball from Drive to local SSD if not already there.
DATASET_TAR = DRIVE_ROOT / 'lebai_duck_pick.tar.gz'
DATASET_DIR = LEROBOT_CACHE / 'local/lebai_duck_pick'

if not DATASET_DIR.exists():
    assert DATASET_TAR.exists(), (
        f'Dataset tarball not found at {DATASET_TAR}.\n'
        f'Run `python convert_local.py` locally, then `tar -czf lebai_duck_pick.tar.gz '
        f'local/lebai_duck_pick` from the result/ dir, and upload the tarball to {DRIVE_ROOT}.'
    )
    LEROBOT_CACHE.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {DATASET_TAR}  ->  {LEROBOT_CACHE}')
    subprocess.run(['tar', '-xzf', str(DATASET_TAR), '-C', str(LEROBOT_CACHE)], check=True)
    print('Extracted.')
else:
    print(f'Dataset already on local SSD: {DATASET_DIR}')

print(f'HF_LEROBOT_HOME = {os.environ["HF_LEROBOT_HOME"]}  (local SSD)')
print(f'CHECKPOINT_DIR  = {CHECKPOINT_DIR}  (Drive)')

## 3. Imports

`make_policy_features` wraps the dataset's plain feature dicts in `PolicyFeature` objects — required by lerobot ≥0.5 (PRD §7.3).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
    from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
    from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
    from lerobot.configs.types import FeatureType, PolicyFeature
except ImportError:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
    from lerobot.common.policies.diffusion.modeling_diffusion import DiffusionPolicy
    from lerobot.common.policies.diffusion.configuration_diffusion import DiffusionConfig
    from lerobot.configs.types import FeatureType, PolicyFeature


def make_policy_features(features_dict):
    """Wrap dataset features into PolicyFeature objects (required by lerobot 0.5+).

    The dataset stores image schemas HWC (e.g. (480, 640, 3)), but DiffusionPolicy's
    DiffusionRgbEncoder reads `images_shape[0]` as the channel count when building
    its dummy input — so for VISUAL features we must transpose to CHW. Other
    lerobot policies (ACT, sarm) accept HWC; the convention is inconsistent inside
    lerobot 0.5.1. See PRD §7 for the full story.
    """
    out = {}
    for name, spec in features_dict.items():
        shape = tuple(spec['shape'])
        if name.startswith('observation.images.'):
            ft = FeatureType.VISUAL
            if len(shape) == 3 and shape[-1] in (1, 3):
                shape = (shape[2], shape[0], shape[1])    # HWC -> CHW
        elif name == 'observation.state':
            ft = FeatureType.STATE
        elif name == 'action':
            ft = FeatureType.ACTION
        else:
            continue
        out[name] = PolicyFeature(type=ft, shape=shape)
    return out


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: no GPU — training will be very slow. Runtime → Change runtime type → GPU.')


## 4. Load dataset metadata

Quick sanity check that the converter wrote what we expect.

In [ ]:
DATASET_REPO_ID = 'local/lebai_duck_pick'  # must match REPO_ID from the converter

dataset_meta = LeRobotDataset(DATASET_REPO_ID).meta
print(f'  episodes: {dataset_meta.total_episodes}')
print(f'  frames:   {dataset_meta.total_frames}')
print(f'  fps:      {dataset_meta.fps}')
print(f'  tasks:    {len(dataset_meta.tasks)}')
print('Features:')
for name, spec in dataset_meta.features.items():
    print(f'  {name}: dtype={spec["dtype"]}, shape={spec.get("shape")}')

fps = dataset_meta.fps

## 5. Sanity-check the dataset

Highest-ROI check in the pipeline — converter bugs (wrong channel, swapped joints, action == state) are nearly invisible from training loss alone but obvious here. If state and action overlap perfectly on every frame, the converter probably fell back to `next_row.jp*` for every action; verify `tgt_jp*` columns are populated in your raw CSVs.

In [ ]:
ds_inspect = LeRobotDataset(DATASET_REPO_ID)
sample = ds_inspect[0]
image_keys = [k for k in sample.keys() if k.startswith('observation.images.')]

fig, axes = plt.subplots(1, len(image_keys), figsize=(5 * len(image_keys), 5))
if len(image_keys) == 1:
    axes = [axes]
for ax, key in zip(axes, image_keys):
    img = sample[key].permute(1, 2, 0).cpu().numpy()
    ax.imshow(img); ax.set_title(key.replace('observation.images.', '')); ax.axis('off')
plt.tight_layout(); plt.show()

print(f"state:  {sample['observation.state'].numpy().round(3)}")
print(f"action: {sample['action'].numpy().round(3)}")
print(f"task:   {sample['task']!r}")

In [ ]:
ep0 = ds_inspect.meta.episodes[0]
ep0_from = int(ep0['dataset_from_index'])
ep0_to   = int(ep0['dataset_to_index'])
frames = [ds_inspect[i] for i in range(ep0_from, ep0_to)]

states  = torch.stack([f['observation.state'] for f in frames]).numpy()
actions = torch.stack([f['action'] for f in frames]).numpy()
t = np.arange(len(states)) / fps
arm_dims = states.shape[1] - 1 if states.shape[1] == 7 else states.shape[1]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for i in range(arm_dims):
    axes[0, 0].plot(t, states[:, i], label=f'j{i}')
    axes[0, 1].plot(t, actions[:, i], label=f'j{i}')
if states.shape[1] == 7:
    axes[1, 0].plot(t, states[:, -1], color='black', label='gripper')
    axes[1, 1].plot(t, actions[:, -1], color='black', label='gripper')
axes[0, 0].set_title('state — arm joints'); axes[0, 0].legend(loc='upper right')
axes[0, 1].set_title('action — arm joints'); axes[0, 1].legend(loc='upper right')
axes[1, 0].set_title('state — gripper amplitude'); axes[1, 0].set_xlabel('time (s)')
axes[1, 1].set_title('action — gripper amplitude'); axes[1, 1].set_xlabel('time (s)')
for a in axes.flat: a.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Configure Diffusion Policy

Overrides from `DiffusionConfig` defaults per PRD §6 (LeRobot's defaults assume bimanual ALOHA at 50 Hz):

| Setting | Value | Why |
| --- | --- | --- |
| `n_obs_steps` | 2 | DP standard — two-frame observation history |
| `horizon` | 16 | Predict 16 actions per forward pass (~1.6 s at 10 Hz) |
| `n_action_steps` | 8 | Execute first 8 (~0.8 s) before re-querying |
| `num_train_timesteps` | 100 | DDPM schedule length |
| `num_inference_steps` | 10 | DDIM at inference, fits 100 ms tick budget |
| `crop_shape` | `(84, 84)` | DP-style random crop |
| `resize_shape` | `(96, 128)` | **Not in PRD — added so the (84,84) crop is meaningful.** 480×640 input random-cropped to 84×84 captures ~2.3% of pixels; pre-resize to 96×128 (preserves 4:3 aspect) makes the crop ratio match the DP paper's PushT (96×96 → 84×84). Revisit if loss won't decrease in the smoke test. |
| `optimizer_lr` | 1e-4 | Same as ACT |

**`delta_timestamps`** — DP needs both past observations (n_obs_steps frames back) and future actions (horizon frames forward). The dataset yields these slices automatically when configured below.

In [ ]:
N_OBS_STEPS    = 2
HORIZON        = 16
N_ACTION_STEPS = 8

# Past observations: frames at t-1 and t  (n_obs_steps=2)
# Future actions:    frames at t, t+1, ..., t+horizon-1
obs_dt    = [(t - (N_OBS_STEPS - 1)) / fps for t in range(N_OBS_STEPS)]
action_dt = [t / fps for t in range(HORIZON)]

delta_timestamps = {
    'observation.state':       obs_dt,
    'observation.images.base': obs_dt,
    'action':                  action_dt,
}
if 'observation.images.wrist' in dataset_meta.features:
    delta_timestamps['observation.images.wrist'] = obs_dt

dataset = LeRobotDataset(DATASET_REPO_ID, delta_timestamps=delta_timestamps)
print(f'Training frames: {len(dataset)} (each yields {N_OBS_STEPS} obs + {HORIZON} actions)')

cfg = DiffusionConfig(
    n_obs_steps=N_OBS_STEPS,
    horizon=HORIZON,
    n_action_steps=N_ACTION_STEPS,
    vision_backbone='resnet18',
    resize_shape=(96, 128),
    crop_shape=(84, 84),
    crop_is_random=True,
    noise_scheduler_type='DDPM',
    num_train_timesteps=100,
    num_inference_steps=10,
    prediction_type='epsilon',
    optimizer_lr=1e-4,
)
all_feats = make_policy_features(dataset.features)
cfg.input_features  = {k: v for k, v in all_feats.items() if k != 'action'}
cfg.output_features = {'action': all_feats['action']}

policy = DiffusionPolicy(cfg, dataset_stats=dataset.meta.stats)
policy.to(device); policy.train()

n_params = sum(p.numel() for p in policy.parameters())
print(f'Policy: {n_params/1e6:.1f}M parameters')

## 7. Optionally resume from a Drive checkpoint

Colab can disconnect after a few hours — DP at 100k steps takes longer than that on T4. This cell finds the most recent `step_*` checkpoint under `CHECKPOINT_DIR` and loads it. Skip if you want to start fresh.

In [ ]:
RESUME = True

start_step = 0
if RESUME:
    ckpts = sorted(CHECKPOINT_DIR.glob('step_*'))
    if ckpts:
        latest = ckpts[-1]
        print(f'Resuming from {latest}')
        policy = DiffusionPolicy.from_pretrained(latest)
        policy.to(device); policy.train()
        start_step = int(latest.name.split('_')[1])
        print(f'Resumed at step {start_step}')
    else:
        print('No checkpoint found — training from scratch.')
else:
    print('RESUME=False — training from scratch.')

## 8. Training loop

`SMOKE_TEST=True` runs only 200 steps so you can confirm loss decreases before committing to a full 100k-step run. Always smoke-test after any config change.

In [ ]:
SMOKE_TEST = False     # flip True for a 200-step sanity run

BATCH_SIZE = 8         # raise to 16 on A100
NUM_STEPS  = 200 if SMOKE_TEST else 100_000
LOG_EVERY  = 50 if SMOKE_TEST else 200
SAVE_EVERY = 100 if SMOKE_TEST else 10_000   # DP checkpoints are bigger than ACT — save less often

dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=device.type == 'cuda', drop_last=True,
)
def cycle(loader):
    while True:
        for batch in loader:
            yield batch
step_iter = cycle(dataloader)

optimizer = torch.optim.AdamW(
    policy.parameters(),
    lr=cfg.optimizer_lr,
    betas=cfg.optimizer_betas,
    eps=cfg.optimizer_eps,
    weight_decay=cfg.optimizer_weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_STEPS)
for _ in range(start_step):
    scheduler.step()

print(f'Will run steps {start_step} → {NUM_STEPS}  bs={BATCH_SIZE}  -> {CHECKPOINT_DIR}')
if SMOKE_TEST:
    print('SMOKE_TEST=True — 200 steps. Confirm loss is decreasing before disabling this flag.')

In [ ]:
loss_history = []
t0 = time.time()

for step in range(start_step, NUM_STEPS):
    batch = next(step_iter)
    batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
             for k, v in batch.items()}

    # DP's forward returns (loss, None) — unlike ACT which returns (loss, loss_dict).
    # See PRD §7.4. We discard the second element.
    loss, _ = policy.forward(batch)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
    optimizer.step()
    scheduler.step()

    loss_history.append(loss.item())

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        rate = (step - start_step + 1) / max(elapsed, 1e-6)
        print(f'step {step:6d}  loss={loss.item():.4f}  '
              f'lr={scheduler.get_last_lr()[0]:.2e}  ({rate:.1f} step/s)')

    if (step + 1) % SAVE_EVERY == 0:
        ckpt_path = CHECKPOINT_DIR / f'step_{step+1:06d}'
        policy.save_pretrained(ckpt_path)
        print(f'  -> saved {ckpt_path}')

final_ckpt = CHECKPOINT_DIR / ('smoke' if SMOKE_TEST else 'final')
policy.save_pretrained(final_ckpt)
print(f'\nDone. Final: {final_ckpt}')


## 9. Loss curve

DP loss is the simple MSE between predicted and target noise (no separate L1/KLD like ACT). Should trend down smoothly — lots of noise tick-to-tick is normal.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_history, alpha=0.4, label='raw')
window = max(1, len(loss_history) // 100)
if len(loss_history) >= window:
    smoothed = np.convolve(loss_history, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(loss_history)), smoothed, label=f'smoothed (w={window})')
ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title('Diffusion Policy training loss'); plt.tight_layout(); plt.show()

## Next

`CHECKPOINT_DIR/final/` is persisted to Drive. Download it to a machine on the robot's LAN and point `run_inference.py --checkpoint` at it. Inference cannot run on Colab — it needs LAN access to the Lebai SDK and the camera service.

```bash
# On the robot machine
python run_inference.py --checkpoint ./checkpoints/dp_run01/final --dry-run
SAFETY_OK=1 python run_inference.py --checkpoint ./checkpoints/dp_run01/final --duration 30
```